In [ ]:
#%matplotlib inline
%matplotlib qt5
 

import os
import pandas as pd
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import mne
import PyQt5
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

from mne.stats import permutation_cluster_test as pcluster_test

In [ ]:
roi =  ['Fz', 'FCz', 'Cz']
#frontal = ['Fp1','Fp2','AFz','AF7', 'AF8','AF3','AF4','Fz','F1','F2','F3','F4','F5','F6','F7','F8','FCz','FC1','FC2','FC3','FC4','FC5','FC6','Cz','C1','C2','C3','C4','C5','C6']

In [ ]:
all_epochs = [] 
data_folder = r"/Users/alexmedvedev/Desktop/Vishka/old_good_montage/all_epochs_offset_tfr" # all_epochs_onset_tfr
for file in os.listdir(data_folder):
    if file.endswith(".fif"):
        filepath = os.path.join(data_folder, file)
        print(f"Loading {filepath}")
        epochs = mne.read_epochs(filepath, preload=True)
        epochs.info['bads'] = []
        epochs.pick_channels(roi)
        all_epochs.append(epochs)

epochs_combined = mne.concatenate_epochs(all_epochs)
epochs_combined

In [ ]:
freqs = np.arange(1, 99)  # or 45
vmin, vmax = -1, 3  
baseline = (-0.2, 0) # normal: (-1, -0.2)
cnorm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)  
tmin, tmax = -0.2, 1 # or 1, 3
kwargs = dict(n_permutations=500, step_down_p=0.05, seed=1, buffer_size=None, out_type="mask")  

tfr = epochs_combined.compute_tfr(
    method="multitaper",
    picks = roi,
    freqs=freqs,
    n_cycles=freqs / 2,
    use_fft=True,
    return_itc=False,
    average=False,
    decim=3, #6 or 3
    time_bandwidth=4.0  
)

tfr.apply_baseline(baseline=baseline, mode="percent")
tfr.crop(tmin=tmin, tmax=tmax)
tfr

In [ ]:
from mne.time_frequency import write_tfrs

In [ ]:
# Save the TFR data to a file
path_to_tfr = Path("/Users/alexmedvedev/Desktop/Vishka/tfr")
output_file = os.path.join(path_to_tfr, "tfr_offset.h5") # onset
write_tfrs(output_file, [tfr])

In [ ]:
selected_channels = ["Fz", "FCz", "Cz"]

channel_indices = mne.pick_channels(tfr.ch_names, include=selected_channels)

event_ids = ["5-6", "7-8", "9-10", "11-12"]

for event in event_ids:
    tfr_ev = tfr[event]  
    avg_tfr = tfr_ev.average()  

    fig, axes = plt.subplots(
        1, len(selected_channels) + 1, figsize=(12, 4), gridspec_kw={"width_ratios": [10] * len(selected_channels) + [1]}
    )

    for idx, (ch_idx, ax) in enumerate(zip(channel_indices, axes[:-1])):  
        _, clusters, p_values, _ = pcluster_test(
            tfr_ev.data[:, ch_idx, :, :],  
            tail=0,  
            threshold=None,
            **kwargs
        )

        cluster_mask = np.zeros(avg_tfr.data.shape[1:], dtype=bool)  

        if clusters:
            for i, cluster in enumerate(clusters):
                if p_values[i] <= 0.05:  
                    time_mask = np.zeros(avg_tfr.data.shape[2], dtype=bool)
                    time_mask[cluster] = True
                    cluster_mask |= time_mask[np.newaxis, :]  

        avg_tfr.plot(
            picks=[ch_idx],  
            cmap="RdBu_r",
            cnorm=cnorm,
            axes=ax,
            colorbar=False,
            show=False,
            mask=cluster_mask,
            mask_style="mask",
        )

        ax.set_title(avg_tfr.ch_names[ch_idx], fontsize=10)
        ax.axvline(0, linewidth=1, color="black", linestyle=":")  
        if idx != 0:
            ax.set_ylabel("")
            ax.set_yticklabels("")

    fig.colorbar(axes[0].images[-1], cax=axes[-1]).ax.set_yscale("linear")
    fig.suptitle(f"ERDS ({event})")
    plt.show()

In [ ]:
event_ids = ["5-6", "7-8", "9-10", "11-12"]
for event in event_ids:
    tfr_ev = tfr[event]
    print(event, tfr_ev)

In [ ]:
# Convert TFR objects to dataframes and add a 'condition' column for each
df_5 = tfr['5-6'].to_data_frame(time_format=None)
df_5['condition'] = '5-6'

df_7 = tfr['7-8'].to_data_frame(time_format=None)
df_7['condition'] = '7-8'

df_9 = tfr['9-10'].to_data_frame(time_format=None)
df_9['condition'] = '9-10'

df_11 = tfr['11-12'].to_data_frame(time_format=None)
df_11['condition'] = '11-12'

# Concatenate all dataframes into one
df_tfr = pd.concat([df_5, df_7, df_9, df_11], ignore_index=True)

In [ ]:
df_tfr

In [ ]:
df_5 = tfr['5-6'].to_data_frame(time_format=None, long_format=True)
df_5['condition'] = '5-6'

df_7 = tfr['7-8'].to_data_frame(time_format=None, long_format=True)
df_7['condition'] = '7-8'

df_9 = tfr['9-10'].to_data_frame(time_format=None, long_format=True)
df_9['condition'] = '9-10'

df_11 = tfr['11-12'].to_data_frame(time_format=None, long_format=True)
df_11['condition'] = '11-12'

# Concatenate all dataframes into one
df_tfr = pd.concat([df_5, df_7, df_9, df_11], ignore_index=True)

In [ ]:
# Map to frequency bands:
freq_bounds = {"_": 0, "delta": 3, "theta": 7, "alpha": 13, "beta": 25, "gamma": 44}
df_tfr["band"] = pd.cut(
    df_tfr["freq"], list(freq_bounds.values()), labels=list(freq_bounds)[1:]
)

# Filter to retain only relevant frequency bands:

#freq_bands_of_interest = ["delta", "theta", "alpha", "beta"]
#df_tfr = df_tfr[df_tfr.band.isin(freq_bands_of_interest)]
#df_tfr["band"] = df_tfr["band"].cat.remove_unused_categories()

In [ ]:
df_tfr

In [ ]:
df_new = df_tfr[['condition', 'freq', 'time', 'channel', 'value']]

In [ ]:
df_5 = df_new[df_new['condition'] == '5-6'].copy()
df_7 = df_new[df_new['condition'] == '7-8'].copy()
df_9 = df_new[df_new['condition'] == '9-10'].copy()
df_11 = df_new[df_new['condition'] == '11-12'].copy()

In [ ]:
df_5

In [ ]:
df_7

In [ ]:
# НА СКОЛЬКО Я ПОНЯЛ С ПОМОЩЬЮ ЭТОЙ ДАТЫ МОЖНО РИСОВАТЬ ГРАФИК ПОПАРНЫХ СРАВНЕНИЙ, НА КОТОРОМ БУДУТ ВИДНЫ ТОЛЬКО "ОТЛИЧАЮЩИЕСЯ" У ДВУХ ГРУПП ЧАСТИ

# Group by freq, time, channel and compute the mean for each condition
df_5_mean = df_5.groupby(['freq', 'time', 'channel'])['value'].mean().reset_index()
df_7_mean = df_7.groupby(['freq', 'time', 'channel'])['value'].mean().reset_index()

# Rename value column
df_5_mean = df_5_mean.rename(columns={'value': 'value_f'})
df_7_mean = df_7_mean.rename(columns={'value': 'value_t'})

# Merge the two DataFrames on freq, time, and channel
df_5_7_diff = df_5_mean.merge(df_7_mean, on=['freq', 'time', 'channel']) # ПЕРВОЕ ВЫЧИТАЕТСЯ ИЗ ВТОРОГО (ТО ЕСТЬ ГРАФИК 7-5) # ПОЗИТИВНЫЕ ЗНАЧЕНИЯ (КРАСНЫЕ НА ГРАФИКЕ) - ЭТО ТЕ, В КОТОРЫЙ 7 БОЛЬШЕ (ТО, ИЗ ЧЕГО ВЫЧИТАЮТ) # НЕГАТИВНЫЕ ЗНАЧЕНИЯ (СИНИЕ НА ГРАФИКЕ) - ЭТО ТЕ, В КОТОРЫЙ 5 БОЛЬШЕ (ТО, ЧТО ВЫЧИТАЮТ)

# Compute difference
df_5_7_diff['value_diff'] = df_5_7_diff['value_t'] - df_5_7_diff['value_f']

# Keep relevant columns
df_5_7_diff = df_5_7_diff[['freq', 'time', 'channel', 'value_diff']]

print(df_5_7_diff.head())

In [ ]:
# НА СКОЛЬКО Я ПОНЯЛ С ПОМОЩЬЮ ЭТОЙ ДАТЫ МОЖНО РИСОВАТЬ ГРАФИК ПОПАРНЫХ СРАВНЕНИЙ, НА КОТОРОМ БУДУТ ВИДНЫ ТОЛЬКО "ОТЛИЧАЮЩИЕСЯ" У ДВУХ ГРУПП ЧАСТИ

# Group by freq, time, channel and compute the mean for each condition
df_9_mean = df_9.groupby(['freq', 'time', 'channel'])['value'].mean().reset_index()
df_11_mean = df_11.groupby(['freq', 'time', 'channel'])['value'].mean().reset_index()

# Rename value column
df_9_mean = df_9_mean.rename(columns={'value': 'value_f'})
df_11_mean = df_11_mean.rename(columns={'value': 'value_t'})

# Merge the two DataFrames on freq, time, and channel
df_9_11_diff = df_9_mean.merge(df_11_mean, on=['freq', 'time', 'channel']) # ПЕРВОЕ ВЫЧИТАЕТСЯ ИЗ ВТОРОГО (ТО ЕСТЬ ГРАФИК 7-5) # ПОЗИТИВНЫЕ ЗНАЧЕНИЯ (КРАСНЫЕ НА ГРАФИКЕ) - ЭТО ТЕ, В КОТОРЫЙ 7 БОЛЬШЕ (ТО, ИЗ ЧЕГО ВЫЧИТАЮТ) # НЕГАТИВНЫЕ ЗНАЧЕНИЯ (СИНИЕ НА ГРАФИКЕ) - ЭТО ТЕ, В КОТОРЫЙ 5 БОЛЬШЕ (ТО, ЧТО ВЫЧИТАЮТ)

# Compute difference
df_9_11_diff['value_diff'] = df_9_11_diff['value_t'] - df_9_11_diff['value_f']

# Keep relevant columns
df_9_11_diff = df_9_11_diff[['freq', 'time', 'channel', 'value_diff']]

print(df_9_11_diff.head())

In [ ]:
df_5_7_diff

In [ ]:
# Determine the min and max value_diff across all channels to set the same color scale
vmin = df_5_7_diff['value_diff'].min()
vmax = df_5_7_diff['value_diff'].max()

# Plotting for multiple channels with the same scale
channels = df_5_7_diff['channel'].unique()
fig, axes = plt.subplots(nrows=1, ncols=len(channels), figsize=(18, 6))

for i, channel in enumerate(channels):
    df_channel = df_5_7_diff[df_5_7_diff['channel'] == channel]
    df_pivot_channel = df_channel.pivot_table(index='freq', columns='time', values='value_diff')
    sns.heatmap(df_pivot_channel, cmap='coolwarm', cbar=True, annot=False, ax=axes[i], vmin=vmin, vmax=vmax)

    # Invert the y-axis for each subplot
    axes[i].invert_yaxis()

    axes[i].set_title(f'Difference for 5_7 {channel}', fontsize=14)
    axes[i].set_xlabel('Time (s)', fontsize=12)
    axes[i].set_ylabel('Frequency (Hz)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Determine the min and max value_diff across all channels to set the same color scale
vmin = df_9_11_diff['value_diff'].min()
vmax = df_9_11_diff['value_diff'].max()

# Plotting for multiple channels with the same scale
channels = df_9_11_diff['channel'].unique()
fig, axes = plt.subplots(nrows=1, ncols=len(channels), figsize=(18, 6))

for i, channel in enumerate(channels):
    df_channel = df_9_11_diff[df_9_11_diff['channel'] == channel]
    df_pivot_channel = df_channel.pivot_table(index='freq', columns='time', values='value_diff')
    sns.heatmap(df_pivot_channel, cmap='coolwarm', cbar=True, annot=False, ax=axes[i], vmin=vmin, vmax=vmax)

    # Invert the y-axis for each subplot
    axes[i].invert_yaxis()

    axes[i].set_title(f'Difference for 9_11 {channel}', fontsize=14)
    axes[i].set_xlabel('Time (s)', fontsize=12)
    axes[i].set_ylabel('Frequency (Hz)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Order channels for plotting:
df_tfr["channel"] = df_tfr["channel"].cat.reorder_categories(("Fz", "FCz", "Cz"), ordered=True)

g = sns.FacetGrid(df_tfr, row="band", col="channel", margin_titles=True)
g.map(sns.lineplot, "time", "value", "condition", n_boot=10)
axline_kw = dict(color="black", linestyle="dashed", linewidth=0.5, alpha=0.5)
g.map(plt.axhline, y=0, **axline_kw)
g.map(plt.axvline, x=0, **axline_kw)
g.set(ylim=(None, 1.5))
g.set_axis_labels("Time (s)", "ERDS")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.add_legend(ncol=2, loc="lower center")
g.fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.08)

In [ ]:
df_mean = (
    df_tfr.query("time > 0")
    .groupby(["condition", "epoch", "band", "channel"], observed=False)[["value"]]
    .mean()
    .reset_index()
)

df_mean

In [ ]:
df_mean = (
    df_tfr.query("time > 0")
    .groupby(["condition", "epoch", "band", "channel"], observed=False)[["value"]]
    .mean()
    .reset_index()
)

g = sns.FacetGrid(
    df_mean, col="condition", col_order=["5-6", "7-8", "9-10", "11-12"], margin_titles=True
)
g = g.map(
    sns.violinplot,
    "channel",
    "value",
    "band",
    cut=0,
    palette="deep",
    order=["Fz", "FCz", "Cz"],
    hue_order=freq_bands_of_interest,
    linewidth=0.5,
).add_legend(ncol=4, loc="lower center")

g.map(plt.axhline, **axline_kw)
g.set_axis_labels("", "ERDS")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.fig.subplots_adjust(right=0.9)

In [ ]:
# ЭТО КАЖИСЬ ПРОСТО НЕРАБОЧИЙ КОД, РАБОЧИЙ НИЖЕ (там все ровно тоже самое, за исключением одной строчки)

selected_channels = ["Fz", "FCz", "Cz"]
df_filtered = df[["time", "freq"] + selected_channels]
freq_bounds = {"_": 0, "delta": 3, "theta": 7, "alpha": 13, "beta": 35, "gamma": 140}
df_filtered["band"] = pd.cut(df_filtered["freq"], list(freq_bounds.values()), labels=list(freq_bounds)[1:])
freq_bands_of_interest = ["delta", "theta", "alpha", "beta"]
df_filtered = df_filtered[df_filtered["band"].isin(freq_bands_of_interest)]
df_filtered["band"] = df_filtered["band"].cat.remove_unused_categories()  
df_long = df_filtered.melt(id_vars=["time", "freq", "band"], var_name="channel", value_name="value")
    
g = sns.FacetGrid(df_long, row="band", col="channel", margin_titles=True, sharey=False)
g.map(sns.lineplot, "time", "value", ci=None)
    
axline_kw = dict(color="black", linestyle="dashed", linewidth=0.5, alpha=0.5)
g.map(plt.axhline, y=0, **axline_kw)  
g.map(plt.axvline, x=0, **axline_kw)  
    
g.set_axis_labels("Time (s)", "TFR Power (ERDS)")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.add_legend()
g.fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.show()


In [ ]:
# Define the channels and frequency bounds
selected_channels = ["Fz", "FCz", "Cz"]
freq_bounds = {"_": 0, "delta": 3, "theta": 7, "alpha": 13, "beta": 25, "gamma": 45}
df_filtered = df_tfr[["condition", "epoch", "freq", "time", "channel", "value"]].copy()  # Use .copy() to avoid SettingWithCopyWarning

# Categorize the frequency bands
df_filtered.loc[:, "band"] = pd.cut(df_filtered["freq"], list(freq_bounds.values()), labels=list(freq_bounds)[1:])
freq_bands_of_interest = ["delta", "theta", "alpha", "beta"]
df_filtered = df_filtered[df_filtered["band"].isin(freq_bands_of_interest)]
df_filtered["band"] = df_filtered["band"].cat.remove_unused_categories()

# Melt the dataframe to long format for plotting
df_long = df_filtered.melt(id_vars=["time", "freq", "band", "condition", "channel"], 
                           value_vars=["value"],  # Specify the column to melt
                           var_name="ch_type", value_name="tfr_value")  # Rename 'value' to 'tfr_value'

# Create the facet grid plot
g = sns.FacetGrid(df_long, row="band", col="condition", margin_titles=True, sharey=False)
g.map(sns.lineplot, "time", "tfr_value", ci=None)

# Add reference lines
axline_kw = dict(color="black", linestyle="dashed", linewidth=0.5, alpha=0.5)
g.map(plt.axhline, y=0, **axline_kw)  
g.map(plt.axvline, x=0, **axline_kw)

# Adjust axis labels and titles
g.set_axis_labels("Time (s)", "TFR Power (ERDS)")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.add_legend()
g.fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)

# Show the plot
plt.show()